# Similarity Search with ChromaDB

This notebook demonstrates how to store grocery-related text data in a Chroma collection and run a simple similarity search. Run the cells in order from top to bottom.

### What this notebook does
- Creates or opens a Chroma collection
- Adds sample grocery documents with metadata
- Queries the collection for the most similar items


In [19]:
import chromadb
from chromadb.utils import embedding_functions

# Step 2: Create the embedding function

This cell loads a sentence-transformer model that converts text into embeddings for similarity matching.

In [21]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Step 3: Create or open the collection

This cell creates a Chroma collection for storing the grocery documents. If the collection already exists, it is reused safely.

In [22]:
client = chromadb.Client()
collection_name = "my_grocery_basket"

collection = client.get_or_create_collection(
    name=collection_name,
    metadata={"description": "A collection for storing grocery data"},
    embedding_function=ef
)
print(f"Collection ready: {collection.name}")

Collection ready: my_grocery_basket


# Step 4: Prepare sample grocery documents

These are example documents that will be stored in the collection for similarity searches.

In [ ]:
texts = [
    'fresh red apples',
    'organic bananas',
    'ripe mangoes',
    'whole wheat bread',
    'farm-fresh eggs',
    'natural yogurt',
    'frozen vegetables',
    'grass-fed beef',
    'free-range chicken',
    'fresh salmon fillet',
    'aromatic coffee beans',
    'pure honey',
    'golden apple',
    'red fruit'
]

# Step 5: Create unique IDs for each document

Each document needs a unique identifier so it can be retrieved and updated later.

In [23]:
ids = [f"food_{index + 1}" for index, _ in enumerate(texts)]
ids

['food_1',
 'food_2',
 'food_3',
 'food_4',
 'food_5',
 'food_6',
 'food_7',
 'food_8',
 'food_9',
 'food_10',
 'food_11',
 'food_12',
 'food_13',
 'food_14']

# Step 6: Add documents to the collection

This cell inserts the sample documents into Chroma with metadata for later filtering or inspection.

In [24]:
collection.add(
    documents=texts,
    ids=ids,
    metadatas=[{"source": "grocery_store", "category": "food"} for _ in texts]
)

# Step 7: Inspect the stored items

This cell fetches the documents back from the collection so you can confirm that they were stored correctly.

In [25]:
all_items = collection.get()
print("Collection contents:")
print(f"Number of documents: {len(all_items['documents'])}")

Collection contents:
Number of documents: 14


# Step 8: Run a similarity search

This function queries the collection for items similar to a given text, such as "apple".

In [26]:
def perform_similarity_search(collection, all_items):
    try:
        query_term = "apple"
        results = collection.query(
            query_texts=[query_term],
            n_results=3
        )
        print(f"Query results for '{query_term}':")
        print(results)

        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            print(f'No documents found similar to "{query_term}"')
            return

        print(f'Top 3 similar documents to "{query_term}":')
        for i in range(min(3, len(results['ids'][0]))):
            doc_id = results['ids'][0][i]
            score = results['distances'][0][i]
            text = results['documents'][0][i]
            if not text:
                print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
            else:
                print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')

    except Exception as error:
        print(f"Error in similarity search: {error}")

# Step 9: Run the full example

This final cell calls the similarity search function so you can see the results in one place.

In [27]:
def main():
    try:
        collection = client.get_or_create_collection(
            name=collection_name,
            metadata={"description": "A collection for storing grocery data."},
            embedding_function=ef
        )
        perform_similarity_search(collection, all_items)
        print(f"Collection ready: {collection.name}")
    except Exception as error:
        print(f"Error: {error}")


if __name__ == "__main__":
    main()

Query results for 'apple':
{'ids': [['food_13', 'food_1', 'food_14']], 'embeddings': None, 'documents': [['golden apple', 'fresh red apples', 'red fruit']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'food', 'source': 'grocery_store'}, {'category': 'food', 'source': 'grocery_store'}, {'category': 'food', 'source': 'grocery_store'}]], 'distances': [[0.3824648857116699, 0.480892539024353, 0.5965152382850647]]}
Top 3 similar documents to "apple":
 - ID: food_13, Text: "golden apple", Score: 0.3825
 - ID: food_1, Text: "fresh red apples", Score: 0.4809
 - ID: food_14, Text: "red fruit", Score: 0.5965
Collection ready: my_grocery_basket
